# PySpark 2: reading, SQL and cleaning

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-02-reading-sql-and-cleaning.ipynb)

A vendor delivers order files with inconsistent status text, malformed values, and an optional column
introduced halfway through a rollout. Your job is to turn those deliveries into a predictable input for
reporting.

This module reads files with declared schemas, routes malformed rows aside, exposes the result to Spark SQL,
and standardizes fields with column expressions. It ends with type casting, null handling, and combining two
schema versions without losing the meaning of a missing value. Keep one question in mind throughout: can you
explain where each null came from?

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** [PySpark 1: foundations](pyspark-01-foundations.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## Read files with a declared schema

Schema inference reads the data an extra time to guess types, and the guess can change with a small sample,
a malformed record or a new partition. A declared schema (a `StructType` or a DDL string) skips that pass,
decodes straight into the contract's types, and turns a type change into a visible problem.

The `mode` option decides what happens to a row that does not fit:

| mode | bad row |
| --- | --- |
| `PERMISSIVE` (default) | kept; fields that fail become null, the raw text goes to the corrupt-record column |
| `DROPMALFORMED` | silently dropped |
| `FAILFAST` | the read fails |

Every file source also exposes a hidden `_metadata` column with the file path, name, size and modification
time, which is the cheapest way to record lineage.

**Where it matters**
- **Vendor deliveries:** one row with `amount` as `"N/A"` is quarantined instead of failing the whole day.
- **Large JSON or CSV:** skipping inference saves a full extra scan of the input.
- **Lineage:** storing `_metadata.file_name` on every row tells an operator which delivery a bad value came from.

**Watch out for**
- Under `PERMISSIVE`, a mismatched type becomes null rather than an error. Count corrupt records (or nulls) after every load.
- `DROPMALFORMED` hides data loss; use it only when the owner agrees that loss is acceptable.
- Spark refuses a query on a raw CSV or JSON read that references only the corrupt-record column. Cache the parsed frame first.
- Set `timestampFormat` and `dateFormat` explicitly when the source does not use ISO 8601.

### Example: JSON with a corrupt-record column

Land a small JSON file in which one row delivers `amount` as `"N/A"`, then read it with a declared schema and
a corrupt-record column. Two rows parse; one is kept aside with its raw text.

In [ ]:
import json

from pyspark.sql import types as T

raw_dir = os.path.join(WORK_DIR, "raw", "orders")
os.makedirs(raw_dir, exist_ok=True)
raw_rows = [
    {"order_id": "o1", "customer_id": "c1", "order_ts": "2026-10-04T09:15:00", "status": "paid", "amount": 120.00},
    {"order_id": "o2", "customer_id": "c2", "order_ts": "2026-10-04T10:05:00", "status": "shipped", "amount": 84.50},
    {"order_id": "o3", "customer_id": "c2", "order_ts": "2026-10-04T11:40:00", "status": "paid", "amount": "N/A"},
]
with open(os.path.join(raw_dir, "part-0.json"), "w") as handle:
    handle.write("\n".join(json.dumps(row) for row in raw_rows))

order_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.DecimalType(12, 2), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

orders_raw = (
    spark.read
    .schema(order_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(raw_dir)
    .cache()  # Spark refuses queries that touch only the corrupt-record column of an uncached read
)

good_orders = orders_raw.where(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
bad_orders = orders_raw.where(F.col("_corrupt_record").isNotNull())
good_orders.show()
bad_orders.select("_corrupt_record").show(truncate=False)
print("landed:", len(raw_rows), "| parsed:", good_orders.count(), "| corrupt:", bad_orders.count())

**What to notice:** Two rows parse into the declared types. Order `o3` does not fit `decimal(12,2)`, so its whole raw line is
kept in `_corrupt_record` instead of failing the read or disappearing. Landed rows equal parsed plus corrupt
rows: that reconciliation is the check to run after every load.

### Example: CSV with quoting and file lineage

CSV needs explicit header, quote and escape settings, and a format for its timestamps. The second email
contains a comma inside quotes. `_metadata.file_name` records which file each row came from.

In [ ]:
csv_dir = os.path.join(WORK_DIR, "raw", "customers")
os.makedirs(csv_dir, exist_ok=True)
with open(os.path.join(csv_dir, "part-0.csv"), "w") as handle:
    handle.write(
        "customer_id,email,created_at\n"
        'c1,"ada@example.com",03/01/2026 09:00\n'
        'c2,"grace, hopper@example.org",04/01/2026 10:30\n'
    )

customers_csv = (
    spark.read
    .schema("customer_id string, email string, created_at timestamp")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("timestampFormat", "dd/MM/yyyy HH:mm")
    .csv(csv_dir)
    .withColumn("source_file", F.col("_metadata.file_name"))
)
customers_csv.show(truncate=False)

**What to notice:** The quoted comma stays inside `grace, hopper@example.org` instead of splitting the row. `04/01/2026 10:30`
is read as 4 January because of the declared `dd/MM/yyyy` format; without it, the value would not parse.
`source_file` names the file each row came from.

## Read from a database over JDBC

`spark.read.format("jdbc")` reads a table or query from any database with a JDBC driver. By default the whole
read is **one** query in **one** task. To read in parallel, give Spark a numeric or date
`partitionColumn`, its `lowerBound` and `upperBound`, and `numPartitions`; Spark issues one range query per
partition. The bounds only decide the stride; they do not filter rows.

```python
orders_db = (
    spark.read.format("jdbc")
    .option("url", "jdbc:postgresql://orders-db.acme.internal:5432/shop")
    .option("dbtable", "(select * from orders where order_date >= date '2026-10-01') as recent")
    .option("user", dbutils.secrets.get("shop", "user"))          # never hard-code credentials
    .option("password", dbutils.secrets.get("shop", "password"))
    .option("partitionColumn", "order_id")
    .option("lowerBound", 1)
    .option("upperBound", 50_000_000)
    .option("numPartitions", 16)        # 16 parallel range queries
    .option("fetchsize", 10_000)        # rows per round trip
    .load()
)
```

This handbook never connects to a live database, so the example is shown, not run.

**Where it matters**
- **Initial loads and small dimensions** pulled from an operational database.
- **Incremental extracts** with a `where updated_at > :watermark` subquery as `dbtable`.

**Watch out for**
- Without `partitionColumn`, a 50-million-row table is read by one task over one connection.
- `numPartitions` is also the number of concurrent connections. Ask the database owner what the source can take.
- A skewed `partitionColumn` (most ids in one range) gives skewed tasks. Choose an evenly spread key.
- For anything large or repeated, prefer change data capture into files or Kafka over hammering the primary database.

### Example

Because nothing is executed here, the example is the code above. Count the work it creates: 16 partitions
means 16 tasks, 16 range queries of about 3.1 million ids each, and 16 open connections at the database.

## Query the same data with SQL and DataFrames

`createOrReplaceTempView` registers a name for a DataFrame's plan in the session catalog, and `spark.sql`
returns a DataFrame. SQL text and DataFrame calls are parsed into the same kind of logical plan before
Catalyst optimizes them, so equivalent queries usually produce the same physical plan. A global temporary
view lives under `global_temp` and is visible to every session in the application.

**Where it matters**
- **Mixed teams:** analysts write SQL, engineers wrap it in tested Python functions, and both run on the same engine.
- **Table creation:** `create table ... as select` on a catalog such as Unity Catalog (`main.gold.daily_revenue`).

**Watch out for**
- Temporary views are session-scoped. Re-registering a name with new logic can leave another cell querying an old plan.
- Long SQL strings get no linting or completion. Keep them short or move complex logic into DataFrame functions.

### Example

Compute daily revenue in SQL and with DataFrames, compare the rows, and check that both plans share the same
aggregate-and-exchange shape. Then create a table with SQL.

In [ ]:
orders.createOrReplaceTempView("orders")

daily_sql = spark.sql("""
    select to_date(order_ts) as order_date, count(*) as order_count, sum(amount) as revenue
    from orders
    where status in ('paid', 'shipped')
    group by to_date(order_ts)
""")
daily_df = (
    orders.where(F.col("status").isin("paid", "shipped"))
    .groupBy(F.to_date("order_ts").alias("order_date"))
    .agg(F.count("*").alias("order_count"), F.sum("amount").alias("revenue"))
)

daily_sql.orderBy("order_date").show()
daily_df.orderBy("order_date").show()
print("same rows:", sorted(map(tuple, daily_sql.collect())) == sorted(map(tuple, daily_df.collect())))
daily_sql.explain()
daily_df.explain()

spark.sql("drop table if exists gold_daily_revenue")
spark.sql("create table gold_daily_revenue using parquet as select * from orders where status = 'paid'")
spark.sql("show tables").show()

**What to notice:** Both queries return the same rows, and both plans have the same shape: a partial `HashAggregate`, an
`Exchange` by `order_date`, and a final `HashAggregate`. SQL and DataFrame code meet in the same optimizer,
so choose whichever reads better. `orders` appears in `show tables` as a temporary view.

## Clean and standardize columns

Built-in string, date and conditional functions are Catalyst expressions evaluated inside the JVM: `trim`,
`lower`, `regexp_replace`, `regexp_extract`, `split`, `to_timestamp`, and `when`/`otherwise` for business
flags. Wrap a set of them in a plain Python function that takes and returns a DataFrame. The function runs
once on the driver to build the plan, not once per row, so it costs nothing at runtime and is easy to test.

**Where it matters**
- **Join keys:** `" Ada@Example.COM "` and `"ada@example.com"` must become the same value before they can match.
- **Reporting flags:** an `order_size` tier or an `is_completed` flag defined once and reused everywhere.

**Watch out for**
- Reaching for a Python UDF to trim, lowercase or split. The built-in version is shorter, faster and keeps code generation intact.
- `when` without `otherwise` returns null for unmatched rows. Decide whether null or a default is the right answer.
- Order matters in a `when` chain: the first matching branch wins, so test the largest threshold first.

### Example

Normalize emails, country codes and phone numbers, and derive an email domain.

In [ ]:
raw_customers = spark.createDataFrame(
    [
        ("  Ada@Example.COM ", " us", "2026-01-03 09:00:00", "+1 (312) 555-0100"),
        ("grace@example.org", "gb ", "2026-01-04 10:30:00", "020 7946 0018"),
        ("not-an-email", "us", "2026-01-05 11:00:00", None),
    ],
    "email string, country_code string, created_at string, phone string",
)

clean_customers = (
    raw_customers
    .select(
        F.trim(F.lower("email")).alias("email"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.to_timestamp("created_at").alias("created_at"),
        F.regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    )
    .withColumn("email_domain", F.element_at(F.split("email", "@"), -1))
    .where(F.col("email").contains("@"))
)
clean_customers.show(truncate=False)

**What to notice:** Emails are trimmed and lower-cased, country codes upper-cased, and phone numbers reduced to digits.
`not-an-email` has no `@`, so the final filter drops it.

### Example: a reusable transformation

Put reusable rules in a function: normalize status text, derive the order date and size tier.

In [ ]:
def standardize_orders(df):
    return (
        df
        .withColumn("status", F.lower(F.trim("status")))
        .withColumn("order_date", F.to_date("order_ts"))
        .withColumn(
            "order_size",
            F.when(F.col("amount") >= 1000, "enterprise")
            .when(F.col("amount") >= 100, "mid_market")
            .otherwise("small"),
        )
        .withColumn("is_completed", F.col("status").isin("paid", "shipped"))
    )

standardized = standardize_orders(orders)
standardized.select("order_id", "status", "amount", "order_size", "is_completed").orderBy("order_id").show()
sizes = {r["order_size"]: r["count"] for r in standardized.groupBy("order_size").count().collect()}

**What to notice:** Order 9 (1,180.00) is the only `enterprise` order, because the `>= 1000` branch is tested first. The
function ran once to build the plan; only the built-in expressions run on executors.

## Cast types and handle nulls

Spark 4 turns ANSI SQL mode on by default: a plain `cast` of `"N/A"` to a number raises
`CAST_INVALID_INPUT` instead of quietly returning null. `try_cast` (and functions such as `try_to_timestamp`)
return null instead, which is what you want when bad values should be quarantined rather than fail the job.
A null produced by `try_cast` still needs to be counted: it is a data-quality signal, not a value.

For missing values, `coalesce` picks the first non-null of several columns, `fillna` replaces nulls with
defaults, and `dropna(subset=...)` removes rows missing required fields.

**Where it matters**
- **Typing a raw string column** from CSV or JSON into decimals and timestamps in silver.
- **Dimension defaults:** an unknown region becomes `"unknown"` so reports do not drop the row.

**Watch out for**
- Jobs that relied on silent nulls in Spark 3 can fail after an upgrade. That is a correctness fix; use `try_cast` where nulls are intended.
- Filling a numeric null with `0` changes averages and counts. Fill only when the business meaning really is zero.
- Compare raw and typed columns: a value present before the cast and null after it is a rejected value.

### Example

A plain cast fails on `"N/A"`; `try_cast` returns null, and comparing the raw and typed columns finds the
rejected value. Then give customer 4's missing region a default and drop rows missing a required field.

In [ ]:
amounts = spark.createDataFrame(
    [("o1", "120.50"), ("o2", "N/A"), ("o3", None)], "order_id string, amount_text string"
)

spark.sparkContext.setLogLevel("OFF")  # the failure below is expected; keep the executor log quiet
try:
    amounts.select(F.col("amount_text").cast("decimal(12,2)")).collect()
except Exception as exc:  # ANSI mode: the plain cast raises
    cast_error = str(exc).splitlines()[0]
else:
    cast_error = ""
finally:
    spark.sparkContext.setLogLevel("ERROR")
print("plain cast:", cast_error[:110])

typed = amounts.withColumn("amount", F.expr("try_cast(amount_text AS DECIMAL(12,2))"))
typed.show()
null_counts = typed.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in typed.columns]).first()
print("nulls per column:", null_counts.asDict())

rejected = typed.where(F.col("amount_text").isNotNull() & F.col("amount").isNull())
rejected.show()  # present before the cast, null after it: a rejected value

customers.fillna({"region": "unknown"}).where("customer_id = 4").show()
print("customers with a region:", customers.dropna(subset=["region"]).count(), "of", customers.count())

**What to notice:** The plain cast fails with `CAST_INVALID_INPUT`. `try_cast` turns `N/A` into null, and so does the missing
`o3`, so `amount` has two nulls while `amount_text` has one. Comparing the two columns separates "never
delivered" (`o3`) from "delivered but invalid" (`o2`). `fillna` gives customer 4 a default region;
`dropna` would remove it instead.

## Handle additive schema changes

A data contract says which fields are required, what types they have, and what a missing value means. An
**additive change** introduces a field that older files lack. Two tools align versions:
`unionByName(..., allowMissingColumns=True)` matches columns by name and fills absent ones with null, and
`mergeSchema` on a Parquet read combines the schemas of all files it reads.

**Where it matters**
- **A feed adds `currency` after a rollout:** historical files stay readable while you decide how to backfill.
- **Partitioned Parquet written over months** by job versions with slightly different columns.

**Watch out for**
- Alignment does not validate meaning. A change from decimal amounts to strings with currency symbols needs a parsing rule and a quarantine, not a union.
- `mergeSchema` reads every file footer, which is slow on large tables. Prefer a table format (Delta, Iceberg) that records the schema.
- A null in the new column means "not delivered", not "no value". Document which one consumers should assume.

### Example

Union an old record with a newer one whose columns arrived in a different order, then read two Parquet
versions together with `mergeSchema`. Older rows get a null currency in both cases.

In [ ]:
old_orders = spark.createDataFrame([(101, 2500)], "order_id long, amount_cents long")
new_orders = spark.createDataFrame([("USD", 4200, 102)], "currency string, amount_cents long, order_id long")
combined = old_orders.unionByName(new_orders, allowMissingColumns=True)
combined.orderBy("order_id").show()

versions = os.path.join(WORK_DIR, "orders_by_version")
orders.select("order_id", "amount").write.mode("overwrite").parquet(os.path.join(versions, "version=1"))
(orders.select("order_id", "amount", F.lit("USD").alias("currency"))
 .write.mode("overwrite").parquet(os.path.join(versions, "version=2")))

merged = spark.read.option("mergeSchema", "true").parquet(versions)
merged.printSchema()
merged.groupBy("version", "currency").count().orderBy("version").show()

**What to notice:** `unionByName` matched columns by name even though the new frame lists them in another order, and filled the
old row's `currency` with null. `mergeSchema` combined both file versions into one schema: version 1 rows
have a null currency, and the `version` column comes from the directory names.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why declare a schema instead of inferring one?

**A.** Inference costs an extra pass over the data and can guess a different type from one bad row or a new file. A declared schema keeps the contract, reads once, and makes bad rows visible.

**Q2.** A `PERMISSIVE` read finished without errors. Why might data still be missing?

**A.** Fields that fail to parse become null and the raw text goes to the corrupt-record column. Unless you count corrupt records or nulls after the load, the job looks healthy while values are lost.

**Q3.** What does `numPartitions` control on a JDBC read, and what is the risk?

**A.** It sets how many range queries run in parallel over `partitionColumn`, which is also how many connections hit the database at once. Too many can overload the source.

**Q4.** In Spark 4, what is the difference between `cast` and `try_cast` on `"N/A"`?

**A.** With ANSI mode on by default, `cast` raises `CAST_INVALID_INPUT`. `try_cast` returns null, which you must then count as a rejected value.

**Q5.** Why is wrapping transformations in a Python function free at runtime?

**A.** The function runs once on the driver to build the plan. Only the built-in expressions it adds run on executors, so there is no per-row Python.

**Q6.** Does `unionByName(..., allowMissingColumns=True)` make a schema change safe?

**A.** It aligns columns by name and fills missing ones with null, but it does not check types or meaning. Validate required fields and types first, and quarantine incompatible deliveries.

## Key takeaways

- Declare schemas; count corrupt records and nulls after every load.
- Record lineage with `_metadata.file_name`; it is free on every file source.
- SQL and DataFrames compile to the same plan, so use whichever reads better.
- In Spark 4 a bad `cast` raises; use `try_cast` when bad values should be quarantined, then count the nulls.
- Partition JDBC reads, and respect the source database's connection limits.

**Next:** [PySpark 3: joins and aggregations](pyspark-03-joins-and-aggregations.ipynb)